# ReplaySAM Demo

This notebook loads the iron ore tomogram from `data/iron_ore_GOH6_500_1/sim_res_0/tomo_noisy.zarr`, previews a slice, generates candidate point prompts from local distance-transform maxima, and prepares a `SAM2PipelineConfig` for a real SAM2 run.


In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "replaysam").exists():
            return path
    raise RuntimeError("Could not find the ReplaySAM repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

INPUT_VOLUME_PATH = REPO_ROOT / "data" / "iron_ore_GOH6_500_1" / "sim_res_0" / "tomo_noisy.zarr"
OUTPUT_PARENT_DIR = REPO_ROOT / "notebooks" / "demo_outputs"
OUTPUT_PARENT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_PARENT_DIR / ".matplotlib"))

if not INPUT_VOLUME_PATH.exists():
    raise FileNotFoundError(f"Input volume not found: {INPUT_VOLUME_PATH}")

print(f"Repository root: {REPO_ROOT}")
print(f"Input volume:    {INPUT_VOLUME_PATH}")
print(f"Output parent:   {OUTPUT_PARENT_DIR}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from replaysam.pipeline import Pipeline
from replaysam.utils.configs import (
    PromptGeneratorConfig,
    SAM2BackboneConfig,
    SAM2PipelineConfig,
)
from replaysam.utils.io import VolumeReader
from replaysam.utils.prompt_generator import PromptGenerator

plt.rcParams["figure.figsize"] = (6, 6)


## 1. Load the input tomogram


In [ ]:
reader = VolumeReader(INPUT_VOLUME_PATH, chunks=(64, 256, 256))
dask_volume = reader.get_dask_array()
loaded_volume = reader.get_numpy_array()

print(dask_volume)
print(
    f"Loaded NumPy array: shape={loaded_volume.shape}, "
    f"dtype={loaded_volume.dtype}, "
    f"range=({np.nanmin(loaded_volume):.3f}, {np.nanmax(loaded_volume):.3f})"
)


In [ ]:
z_slice = loaded_volume.shape[0] // 2

fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
ax.imshow(loaded_volume[z_slice], cmap="gray")
ax.set_title(f"Input volume, z={z_slice}")
ax.axis("off");


## 2. Generate candidate point prompts


In [ ]:
prompt_config = PromptGeneratorConfig(
    binarisation_threshold=None,
    dist_val_thresh=5.0,
    max_filter_size=7,
    crop_size=(256, 256, 256),
    crop_overlap=(32, 32, 32),
)

prompt_generator = PromptGenerator(tomo=loaded_volume)
prompt_coords, prompt_values = prompt_generator.generate_peak_local_max_prompts(
    crop_size=prompt_config.crop_size,
    crop_overlap=prompt_config.crop_overlap,
    bin_thresh=prompt_config.binarisation_threshold,
    dist_val_thresh=prompt_config.dist_val_thresh,
    max_filter_size=prompt_config.max_filter_size,
    show_progress=True,
    return_max_values=True,
)

print(f"Generated {len(prompt_coords)} prompts")
for idx, (coord, value) in enumerate(zip(prompt_coords[:20], prompt_values[:20], strict=True), start=1):
    print(f"{idx:02d}. zyx={tuple(int(v) for v in coord)}  distance={value:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
ax.imshow(loaded_volume[z_slice], cmap="gray")

if len(prompt_coords):
    near_slice = np.abs(prompt_coords[:, 0] - z_slice) <= 4
    if np.any(near_slice):
        ax.scatter(
            prompt_coords[near_slice, 2],
            prompt_coords[near_slice, 1],
            s=80,
            marker="x",
            c="tab:red",
            linewidths=2,
            label="Prompt within +/-4 z slices",
        )
        ax.legend(loc="upper right")

ax.set_title(f"Prompt locations near z={z_slice}")
ax.axis("off");


## 3. Build a ReplaySAM pipeline config


In [ ]:
pipeline_config = SAM2PipelineConfig(
    volume_path=INPUT_VOLUME_PATH,
    output_parent_dir=OUTPUT_PARENT_DIR / "pipeline_outputs",
    backbone_config=SAM2BackboneConfig(
        model_size="tiny",
        compute_device="cuda:0",
        storage_device="cpu",
        compile=False,
    ),
    prompt_generator_config=prompt_config,
    max_prompts=10,
    inference_axes=(0, 1, 2),
    majority_voting_threshold=2,
    note="Demo config for the iron ore GOH6 tomogram.",
)

display(pipeline_config.as_dict())


The full pipeline creates a SAM2 backbone and expects the model assets and a compatible GPU environment. Keep the cell below disabled until those are available.


In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    pipeline = Pipeline(pipeline_config)
    pipeline.run()
    print(f"Pipeline output: {pipeline.output_path}")
